XGBoost Cricket Score Prediction

This notebook implements an XGBoost regressor to predict the final innings score based on ball-by-ball data.


In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GridSearchCV

# Constants
FILE_PATH = "final_ball_by_ball_first_innings.csv"
OVERS_TO_PREDICT = [5, 10, 15, 20]

1. Data Loading and Sorting

We ensure the data is chronological to correctly calculate cumulative stats.

In [6]:
df = pd.read_csv(FILE_PATH)
df = df.sort_values(by=["match_id", "over", "ball"]).reset_index(drop=True)

2. Feature Engineering

Calculating the state of the match at every ball.

In [7]:
# Cumulative Progress
df["runs_so_far"] = df.groupby("match_id")["runs_total"].cumsum()
df["wickets_so_far"] = df.groupby("match_id")["wicket_fallen"].cumsum()
df["balls_so_far"] = df.groupby("match_id").cumcount() + 1
df["overs_so_far"] = df["balls_so_far"] / 6.0

# Rate and Remaining Resources
df["current_run_rate"] = df["runs_so_far"] / df["overs_so_far"].replace(0, np.nan)
df["balls_remaining"] = 120 - df["balls_so_far"]
df["wickets_remaining"] = 10 - df["wickets_so_far"]

# Target Variable: Final Score of the match
final_score_map = df.groupby("match_id")["runs_total"].sum().reset_index()
final_score_map = final_score_map.rename(columns={"runs_total": "final_score"})
df = df.merge(final_score_map, on="match_id")

3. Categorical Encoding

Encoding categorical features before the split to ensure consistency.

In [8]:
categorical_cols = ["venue", "batter", "bowler", "non_striker"]
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

4. Train/Test Split

We split by `match_id` to prevent data leakage (having balls from the same match in both sets).

In [9]:
feature_cols = [
    "runs_so_far", "wickets_so_far", "balls_so_far", "overs_so_far",
    "current_run_rate", "balls_remaining", "wickets_remaining",
    "venue", "batter", "bowler", "non_striker"
]

np.random.seed(42)
match_ids = df["match_id"].unique()
np.random.shuffle(match_ids)

train_split = int(len(match_ids) * 0.8)
train_matches = match_ids[:train_split]
test_matches = match_ids[train_split:]

train_df = df[df["match_id"].isin(train_matches)]
test_df = df[df["match_id"].isin(test_matches)]

X_train, y_train = train_df[feature_cols], train_df["final_score"]
X_test, y_test = test_df[feature_cols], test_df["final_score"]

5. XGBoost Model Training

In [3]:
model = xgb.XGBRegressor(
    objective="reg:squarederror",
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    random_state=42
)

print("Starting training...")
model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=100)

Starting training...


NameError: name 'X_train' is not defined

5. HYPERPARAMETER TUNING

In [ ]:

# Define the parameter grid to test
param_grid = {
    'max_depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [300, 500],
    'subsample': [0.8],
    'colsample_bytree': [0.8]
}

# Initialize the XGBRegressor
xgb_base = xgb.XGBRegressor(objective="reg:squarederror", tree_method="hist", random_state=42)

# Setup GridSearchCV
grid_search = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid,
    scoring='neg_mean_absolute_error',
    cv=3,
    verbose=1,
    n_jobs=-1
)

print("Starting Hyperparameter Tuning...")
grid_search.fit(X_train, y_train)

# Identify the best model
best_model = grid_search.best_estimator_
print(f"Best Parameters: {grid_search.best_params_}")



Starting Hyperparameter Tuning...
Fitting 3 folds for each of 18 candidates, totalling 54 fits


6. Evaluation and Visualization

Testing accuracy at specific milestones (5, 10, 15, 20 overs).

In [ ]:
all_results = []

for match in test_matches:
    match_data = test_df[test_df["match_id"] == match]
    for over in OVERS_TO_PREDICT:
        ball_idx = over * 6
        snapshot = match_data[match_data["balls_so_far"] >= ball_idx].head(1)
        
        if not snapshot.empty:
            pred = best_model.predict(snapshot[feature_cols])[0]
            actual = snapshot["final_score"].values[0]
            all_results.append({
                "match_id": match,
                "over": over,
                "pred": round(pred),
                "actual": actual,
                "mae": abs(actual - pred)
            })

res_df = pd.DataFrame(all_results)

# Plotting
bins = [0, 10, 20, 30, 40, 50, 100, 200]
labels = ["0–10", "10–20", "20–30", "30–40", "40–50", "50–100", "100+"]
res_df["MAE_bucket"] = pd.cut(res_df["mae"], bins=bins, labels=labels, right=False)
freq = res_df["MAE_bucket"].value_counts().sort_index()

plt.figure(figsize=(10, 5))
freq.plot(kind='bar', color='teal', edgecolor='black')
plt.title("Distribution of Prediction Errors (MAE)")
plt.xlabel("Error Range (Runs)")
plt.ylabel("Frequency")
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.show()

print(f"Overall Test MAE: {res_df['mae'].mean():.2f}")